# Дерево решений — мини-проект

Сквозной мини-проект: один датасет, решения передаются между этапами.


In [ ]:
# Setup
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
import warnings
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import plot_tree
from sklearn.metrics import ConfusionMatrixDisplay

np.random.seed(42)



## Постановка задачи `(intro)`


Классифицируем **сорта вина** по химическому составу (sklearn `load_wine`, 3 класса, 13 признаков).

**Сюжет:** EDA → split → Gini vs entropy → sweep `max_depth` → **ограничение глубины** → визуализация и importance → confusion matrix на **финальном** дереве.


## Загрузка и EDA `(eda)`


In [ ]:
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.05)

wine = load_wine(as_frame=True)
df = wine.frame.copy()
target_col = "target"
feature_cols = list(wine.feature_names)
class_names = list(wine.target_names)

print(f"Объектов: {len(df)}, признаков: {len(feature_cols)}, классов: {df[target_col].nunique()}")
display(df.head())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df[target_col].value_counts().sort_index()
axes[0].bar(class_names, counts.values, color="steelblue", edgecolor="white")
axes[0].set_title("Баланс классов")

for cls in sorted(df[target_col].unique()):
    sub = df[df[target_col] == cls]
    axes[1].scatter(sub["alcohol"], sub["color_intensity"], alpha=0.7, s=35, label=class_names[cls])
axes[1].set_xlabel("alcohol")
axes[1].set_ylabel("color_intensity")
axes[1].set_title("Два разделяющих признака")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## Решение: train / test split `(example)`


**Решение:** стратифицированный split. Деревья не требуют масштабирования.


In [ ]:
X = df[feature_cols].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, test: {len(X_test)}")


## Критерий: Gini vs entropy `(experiment)`


Сравниваем критерии при одинаковой глубине `max_depth=5`.


In [ ]:
crit_rows = []
for crit in ("gini", "entropy"):
    m = DecisionTreeClassifier(criterion=crit, max_depth=5, random_state=42)
    m.fit(X_train, y_train)
    crit_rows.append({
        "criterion": crit,
        "accuracy train": m.score(X_train, y_train),
        "accuracy test": m.score(X_test, y_test),
    })

crit_df = pd.DataFrame(crit_rows).sort_values("accuracy test", ascending=False)
display(crit_df.round(4))


## Решение: выбор criterion `(example)`


**Решение:** берём критерий с лучшим test accuracy — дальше подбираем глубину только для него.


In [ ]:
CHOSEN_CRITERION = crit_df.iloc[0]["criterion"]
print(f"Выбран criterion: {CHOSEN_CRITERION!r}")


## Глубина и переобучение `(experiment)`


Без ограничений дерево запоминает train. Ищем `max_depth` с лучшим test accuracy.


In [ ]:
depths = list(range(1, 16))
train_acc, test_acc = [], []

for d in depths:
    m = DecisionTreeClassifier(criterion=CHOSEN_CRITERION, max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train))
    test_acc.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_acc, "o-", label="train", color="steelblue")
ax.plot(depths, test_acc, "s-", label="test", color="crimson")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.set_title(f"Глубина vs accuracy (criterion={CHOSEN_CRITERION})")
ax.legend()
plt.tight_layout()
plt.show()

BEST_DEPTH = depths[int(np.argmax(test_acc))]
print(f"Лучший max_depth на test: {BEST_DEPTH} (acc={max(test_acc):.3f})")


## Решение: финальное дерево `(example)`


**Решение:** обучаем `final_tree` с выбранными `criterion` и `max_depth`. Все диагностики — на нём.


In [ ]:
final_tree = DecisionTreeClassifier(
    criterion=CHOSEN_CRITERION, max_depth=BEST_DEPTH, random_state=42
)
final_tree.fit(X_train, y_train)
y_pred = final_tree.predict(X_test)

print(f"Финальное дерево: criterion={CHOSEN_CRITERION!r}, max_depth={BEST_DEPTH}")
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))
print(f"Accuracy test: {accuracy_score(y_test, y_pred):.3f}")


## Визуализация дерева `(viz)`


`plot_tree` для `final_tree` — правила «признак < порог» в каждом узле.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    final_tree,
    feature_names=feature_cols,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title(f"final_tree (max_depth={BEST_DEPTH})")
plt.tight_layout()
plt.show()


## Важность признаков `(viz)`


In [ ]:
imp = final_tree.feature_importances_
order = np.argsort(imp)

plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_cols)[order], imp[order], color="steelblue")
plt.xlabel("feature_importances_")
plt.title("Важность признаков (final_tree)")
plt.tight_layout()
plt.show()


## Confusion matrix `(viz)`


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, cmap="Blues", ax=ax
)
ax.set_title(f"Confusion matrix (final_tree)")
plt.tight_layout()
plt.show()


## Чек-лист мини-проекта `(summary)`


1. EDA → один stratified split.
2. Сравнили Gini/entropy → **выбрали** `CHOSEN_CRITERION`.
3. Sweep depth → **ограничили** `BEST_DEPTH` → `final_tree`.
4. `plot_tree`, importance, confusion — на **одном** финальном дереве.
5. График train vs test — признак переобучения без ограничений.
6. Помнить: оси-параллельные границы, нестабильность одного дерева.
